# Cloud Training Runner — Kaggle / Colab

Generic runner for any milestone's `scripts/train/train_milestone_<x>.py`. Works on both
Kaggle Notebooks and Google Colab — set `PLATFORM` in the first cell and the rest adapts.

Full step-by-step context lives in `CLOUD_TRAINING.md` at the repo root — read that first if
anything here is unclear. This notebook is intentionally thin; it should not contain any
training logic itself (that lives in `scripts/train/train_milestone_<x>.py` so it stays
identical whether invoked from here, from Kaggle, or from Colab).

In [ ]:
# ---- CONFIGURE THIS CELL ----
PLATFORM = "kaggle"          # "kaggle" or "colab"
REPO_URL = "https://github.com/<you>/<repo>.git"
MILESTONE = "milestone_b"    # e.g. milestone_b, milestone_c, ... milestone_i
KAGGLE_DATASET_SLUG = "<your-kaggle-username>/<your-dataset-name>"   # only used if PLATFORM == kaggle
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/goal-conditioned-jepa"    # only used if PLATFORM == colab
RESUME_FROM = None            # e.g. "checkpoints/milestone_b/latest.pt", or None for a fresh run
GIT_USER_EMAIL = "you@example.com"
GIT_USER_NAME = "you"

In [ ]:
# ---- GPU CHECK ----
!nvidia-smi

In [ ]:
# ---- MOUNT / ATTACH PERSISTENT STORAGE ----
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(f"{DRIVE_PROJECT_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_PROJECT_DIR}/data/metadit", exist_ok=True)
elif PLATFORM == "kaggle":
    # Attach the dataset via the Kaggle UI (Add Data) before running this cell.
    # This just confirms it's visible.
    !ls /kaggle/input/
else:
    raise ValueError("PLATFORM must be 'kaggle' or 'colab'")

In [ ]:
# ---- CLONE REPO + INSTALL DEPS ----
!git clone {REPO_URL} repo
%cd repo
!pip install -r requirements.txt -q

In [ ]:
# ---- LINK DATA + CHECKPOINTS TO PERSISTENT STORAGE ----
if PLATFORM == "kaggle":
    !ln -sfn /kaggle/input/{KAGGLE_DATASET_SLUG.split('/')[-1]} data/metadit
    # /kaggle/working persists for the session and can be "Saved" as a notebook version
    !mkdir -p /kaggle/working/checkpoints
    !ln -sfn /kaggle/working/checkpoints checkpoints
elif PLATFORM == "colab":
    !ln -sfn {DRIVE_PROJECT_DIR}/data/metadit data/metadit
    !ln -sfn {DRIVE_PROJECT_DIR}/checkpoints checkpoints

In [ ]:
# ---- RUN TRAINING ----
resume_flag = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
!python scripts/train/train_{MILESTONE}.py --config configs/{MILESTONE}.yaml {resume_flag}

## Sync-back checklist (see `CLOUD_TRAINING.md` §4)
- [ ] Checkpoint saved to persistent storage (already true if using the symlinked path above)
- [ ] `checkpoints/<milestone>/REPORT.md` updated with metrics, done-criteria status, platform used
- [ ] Results pushed to GitHub (next cell) or manually copied back before closing this session

In [ ]:
# ---- OPTIONAL: PUSH RESULTS BACK TO GITHUB ----
# Requires a GitHub personal access token.
# On Kaggle: store it as a Kaggle Secret and load it here.
# On Colab: use `from google.colab import userdata; token = userdata.get('GITHUB_TOKEN')`
# then set the remote URL to include the token before pushing.
#
# !git config user.email "{GIT_USER_EMAIL}"
# !git config user.name "{GIT_USER_NAME}"
# !git add checkpoints/{MILESTONE}/
# !git commit -m "{MILESTONE}: cloud training run, see REPORT.md"
# !git push